# P3-2 · Bài 3 · Attention và Transformer

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, math
def attention(q, k, v, mask=None):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))   # (..., T, T)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    return torch.softmax(scores, dim=-1) @ v                    # (..., T, d_k)

In [ ]:
T = 8
mask = torch.tril(torch.ones(T, T))     # tam giác dưới: 1 = được nhìn, 0 = bị che

In [ ]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h = n_heads
        self.qkv = torch.nn.Linear(d_model, 3 * d_model)      # W_Q, W_K, W_V gộp lại
        self.o = torch.nn.Linear(d_model, d_model)            # W_O

    def forward(self, x, causal=False):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)                                   # mỗi cái (B, T, D)
        q, k, v = [t.view(B, T, self.h, D // self.h).transpose(1, 2) for t in (q, k, v)]   # (B, h, T, d_k)
        att = q @ k.transpose(-2, -1) / math.sqrt(D // self.h)                    # (B, h, T, T)
        if causal:
            att = att.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), 1), float("-inf"))
        y = torch.softmax(att, dim=-1) @ v                                        # (B, h, T, d_k)
        return self.o(y.transpose(1, 2).reshape(B, T, D))                         # (B, T, D)

## Bài tập 4: "Nó" là ai: attention trên một câu {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
tu = ["con", "mèo", "đuổi", "con", "chuột", "vì", "nó", "đói"]
# các chiều: [là con vật, là chủ ngữ, là động từ, là đại từ]
X = np.array([[0.1, 0.2, 0, 0], [1, 1, 0, 0], [0, 0, 1, 0], [0.1, 0, 0, 0], [1, 0, 0, 0],
              [0, 0, 0, 0], [0, 0, 0, 1], [0, 0, 0.5, 0]], dtype=float)
Wq = np.zeros((4, 4)); Wq[3, 0] = 2; Wq[3, 1] = 2      # đại từ hỏi: "ai là con vật và là chủ ngữ?"
Wk = np.eye(4) * 2
# TODO: A (8 × 8 trọng số attention), hàng của "nó"

In [ ]:
Q, K = X @ Wq, X @ Wk
S = Q @ K.T / np.sqrt(Q.shape[1])
A = np.exp(S - S.max(1, keepdims=True)); A /= A.sum(1, keepdims=True)
for w, a in zip(tu, A[6]):
    print(f"{w:6s} {a:.3f} {'█' * int(a * 40)}")

con    0.026 █
mèo    0.791 ███████████████████████████████
đuổi   0.014 
con    0.018 
chuột  0.107 ████
vì     0.014 
nó     0.014 
đói    0.014 


## Bài tập 5: Kiểm tra causal mask bằng thí nghiệm {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch
torch.manual_seed(0)
x = torch.randn(1, 6, 8)
x2 = x.clone(); x2[0, -1] = torch.randn(8)       # chỉ đổi token cuối

def attention(x, causal=True):
    T, d = x.shape[1], x.shape[2]
    S = x @ x.transpose(1, 2) / d ** 0.5
    # TODO: nếu causal, che các vị trí j > i bằng -inf
    return torch.softmax(S, dim=-1) @ x

In [ ]:
def attention(x, causal=True):
    T, d = x.shape[1], x.shape[2]
    S = x @ x.transpose(1, 2) / d ** 0.5
    if causal:
        S = S.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1), float("-inf"))
    return torch.softmax(S, dim=-1) @ x

diff_co = (attention(x)[0, :5] - attention(x2)[0, :5]).abs().max().item()
diff_khong = (attention(x, False)[0, :5] - attention(x2, False)[0, :5]).abs().max().item()
print(f"có mask: {diff_co:.2e}   không mask: {diff_khong:.2e}")

có mask: 0.00e+00   không mask: 3.45e-01


## Bài tập 6: Tự cài multi-head attention và so với PyTorch {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
mha = nn.MultiheadAttention(embed_dim=16, num_heads=4, batch_first=True)
x = torch.randn(2, 5, 16)          # batch 2, 5 token, 16 chiều

def my_mha(x, mha):
    B, T, D = x.shape; H = mha.num_heads; dh = D // H
    qkv = x @ mha.in_proj_weight.T + mha.in_proj_bias        # (B, T, 3D)
    q, k, v = qkv.split(D, dim=-1)
    # TODO: (B, T, D) → (B, H, T, dh); attention; ghép về (B, T, D); out_proj
    return x

In [ ]:
def my_mha(x, mha):
    B, T, D = x.shape; H = mha.num_heads; dh = D // H
    qkv = x @ mha.in_proj_weight.T + mha.in_proj_bias
    q, k, v = qkv.split(D, dim=-1)
    q, k, v = [t.view(B, T, H, dh).transpose(1, 2) for t in (q, k, v)]     # (B, H, T, dh)
    A = torch.softmax(q @ k.transpose(-2, -1) / dh ** 0.5, dim=-1)        # (B, H, T, T)
    out = (A @ v).transpose(1, 2).reshape(B, T, D)
    return mha.out_proj(out)

with torch.no_grad():
    ref, _ = mha(x, x, x)
    diff = (my_mha(x, mha) - ref).abs().max().item()
print(f"chênh lệch lớn nhất: {diff:.2e}")

chênh lệch lớn nhất: 0.00e+00


## Bài tập 7: RoPE chỉ quan tâm khoảng cách {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import math
import numpy as np
theta = 0.3
q = np.array([1.0, 0.5]); k = np.array([0.8, -0.2])

def rope(v, m):
    # TODO: xoay vector 2 chiều v một góc m * theta
    return v

def diem(mq, mk):
    return rope(q, mq) @ rope(k, mk)

In [ ]:
def rope(v, m):
    a = m * theta
    R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])
    return R @ v

for mq, mk in [(3, 1), (10, 8), (3, 5), (0, 0)]:
    print(f"vị trí q={mq:2d}, k={mk}: điểm {diem(mq, mk):+.4f} (khoảng cách {mq - mk:+d})")

vị trí q= 3, k=1: điểm +0.2389 (khoảng cách +2)
vị trí q=10, k=8: điểm +0.2389 (khoảng cách +2)
vị trí q= 3, k=5: điểm +0.9165 (khoảng cách -2)
vị trí q= 0, k=0: điểm +0.7000 (khoảng cách +0)


## Bài tập 8: Đếm tham số một model GPT nhỏ {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch.nn as nn

class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x, mask=None):
        a, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x), attn_mask=mask)
        x = x + a
        return x + self.mlp(self.ln2(x))

d, V, T, L = 256, 8000, 512, 6
# TODO: n_block (đếm bằng code), cong_thuc = 12 * d * d, tong (L block + embedding token + embedding vị trí + LayerNorm cuối)

In [ ]:
n_block = sum(p.numel() for p in Block(d, 8).parameters())
cong_thuc = 12 * d * d
tong = L * n_block + V * d + T * d + 2 * d
print(f"một block: {n_block:,} (công thức 12d² = {cong_thuc:,}, chênh do bias và LayerNorm)")
print(f"cả model: {tong:,} tham số ≈ {tong / 1e6:.1f} triệu, embedding chiếm {V * d / tong:.0%}")

một block: 789,760 (công thức 12d² = 786,432, chênh do bias và LayerNorm)
cả model: 6,918,144 tham số ≈ 6.9 triệu, embedding chiếm 30%
